<a href="https://colab.research.google.com/github/CaoTrongNghia/dipoleMoment-dftEstimator/blob/main/scalar_prediction_qm9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install torch_geometric

# Clone TorchMD-NET fresh
!git clone https://github.com/torchmd/torchmd-net.git
%cd torchmd-net

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 85.9 MB/s eta 0:00:00
Cloning into 'torchmd-net'...
remote: Enumerating objects: 8526, done.
remote: Counting objects: 100% (403/403), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 8526 (delta 365), reused 347 (delta 337), pack-reused 8123 (from 3)
Receiving objects: 100% (8526/8526), 188.84 MiB | 17.67 MiB/s, done.
Resolving deltas: 100% (5952/5952), done.
/content/torchmd-net


In [ ]:
# Install TorchMD-NET
!pip install -e . --no-cache-dir --default-timeout=200

# torchvision and torchaudio are not needed

Obtaining file:///content/torchmd-net
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 174.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 423.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 425.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 438.0 MB/s eta 0:00:00
  Building editable for torchmd-net (pyproject.toml) ... done
  Created wheel for torchmd-net: filename=torchmd_net-2.6.1-0.editable-py3-none-any.whl size=9198 sha256=1103fa8c7282a8d69f93359d5cc7eb3798f38d10e180dc3fd309ce9549303a0b
  Stored in directory: /tmp/pip-ephem-wheel-cache-1brsh705/wheels/94/d6/aa/7f61a3dc80be375e41c625be3732c19212bdab05f6d0c

In [ ]:
import sys
sys.path.append("/usr/local/lib/python3.11/site-packages/")
sys.path.append("/content/torchmd-net")

In [ ]:
# === Base imports for QM9 + TorchMD-NET work ===
import torch
from torchmdnet.datasets import QM9
from torchmdnet.models.model import load_model
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


/content/torchmd-net/torchmdnet/datasets/maceoff.py:19: SyntaxWarning: invalid escape sequence '\S'
  energy_re = re.compile("energy=(\S+)")
/content/torchmd-net/torchmdnet/models/utils.py:472: SyntaxWarning: invalid escape sequence '\c'
  \text{Swish}(x) = x \cdot \sigma(\beta x)
/content/torchmd-net/torchmdnet/models/utils.py:492: SyntaxWarning: invalid escape sequence '\o'
  \text{SwiGLU}(x) = \text{Linear}_1(x) \otimes \text{Swish}(\text{Linear}_2(x))


In [ ]:
import os

# 1. Setup the directory
raw_dir = "./data/qm9/raw"
os.makedirs(raw_dir, exist_ok=True)

print("Applying manual fix for QM9 files...")

# --- FIX 1: Create 'atomref.txt' manually ---
# This contains the reference energies for atoms (H, C, N, O, F).
# TorchMD-Net needs this to exist, even if you are just predicting dipoles.
atomref_content = """-0.500273 -0.498857 -0.497912 -0.502080 -0.505342 -0.499446
-37.846772 -37.845355 -37.844411 -37.848580 -37.851842 -37.846202
-54.583857 -54.582445 -54.581501 -54.585670 -54.588932 -54.583021
-75.064579 -75.063166 -75.062222 -75.066391 -75.069653 -75.063853
-99.718730 -99.717317 -99.716373 -99.720542 -99.723804 -99.717808
"""

with open(os.path.join(raw_dir, "atomref.txt"), "w") as f:
    f.write(atomref_content)
print("-> Created atomref.txt")

# --- FIX 2: Create 'uncharacterized.txt' manually ---
# These are specific molecules in QM9 that are known to be bad/failed calculations.
# We must list them so the code knows to skip them.
unchar_content = """21725
87037
59827
128113
112979
"""
with open(os.path.join(raw_dir, "uncharacterized.txt"), "w") as f:
    f.write(unchar_content)
print("-> Created uncharacterized.txt")

# --- FIX 3: Bypass 'qm9.sd' ---
# The code tries to download this large file, but it ACTUALLY uses the data inside 'qm9.zip'.
# We create an empty dummy file so the "check if file exists" function passes.
# REMOVED: This fix was problematic, as it created an empty qm9.sd.
# We will now rely on the QM9 dataset to extract qm9.sd from qm9.zip.

# --- FIX 4: Ensure qm9.zip is there ---
qm9_zip_path = os.path.join(raw_dir, "qm9.zip")
if os.path.exists(qm9_zip_path):
    print("-> qm9.zip found")
else:
    print("-> Downloading qm9.zip...")
    # The URL below is the direct download link from torch-geometric's QM9 dataset source
    !wget -q --show-progress -O {qm9_zip_path} https://www.dropbox.com/s/vr8w62e3l49m01x/qm9.zip?dl=1
    if os.path.exists(qm9_zip_path):
        print("-> qm9.zip downloaded successfully.")
    else:
        print("-> ERROR: qm9.zip download failed.")


Applying manual fix for QM9 files...
-> Created atomref.txt
-> Created uncharacterized.txt
-> qm9.zip found (Good!)


In [ ]:
import os
import shutil # needed for rmtree

from torch_geometric.loader import DataLoader
from torchmdnet.datasets import QM9

# Define the directory to store the dataset
data_dir = "./data/qm9"


raw_dir = os.path.join(data_dir, "raw")
processed_dir = os.path.join(data_dir, "processed")
qm9_sd_path = os.path.join(raw_dir, "qm9.sd") # Path to the dummy file

if os.path.exists(qm9_sd_path):
    print(f"Removing dummy/potentially corrupted '{qm9_sd_path}'...")
    os.remove(qm9_sd_path)

if os.path.exists(processed_dir):
    print(f"Removing existing processed data directory '{processed_dir}'...")
    shutil.rmtree(processed_dir) # Remove processed data to force re-processing

print("Attempting to load QM9 dataset with a clean state...")

# Load the QM9 dataset
dataset = QM9(root=data_dir, label="dipole_moment")

# Split the dataset
train_cutoff = 110000
val_cutoff = 120000

train_dataset = dataset[:train_cutoff]
val_dataset = dataset[train_cutoff:val_cutoff]
test_dataset = dataset[val_cutoff:]

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Used a smaller batch size to avoid VRAM (GPU memory) errors.
BATCH_SIZE = 16

# Create data loaders for batching
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nUsing a batch size of: {BATCH_SIZE}")

Removing existing processed data directory './data/qm9/processed'...
Attempting to load QM9 dataset with a clean state...


Processing...
  4%|▎         | 4804/133885 [00:04<01:59, 1083.18it/s]


AttributeError: 'NoneType' object has no attribute 'GetNumAtoms'

In [ ]:
import torch
from torchmdnet.models.model import TorchMD_Net
from torchmdnet.models.torchmd_et import TorchMD_ET
from torchmdnet.models.output_modules import DipoleMoment

# Make sure 'device' is defined
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Define the GNN (Equivariant Transformer)
representation_model = TorchMD_ET(
    hidden_channels=128,
    num_layers=6,
    num_rbf=64,
    rbf_type="expnorm",
    trainable_rbf=False,
    activation="silu",
    attn_activation="silu",
    neighbor_embedding=True,
    num_heads=8,
    distance_influence="both",
    cutoff_lower=0.0,
    cutoff_upper=5.0
)

# 2. Define the Output Head
output_model = DipoleMoment(
    hidden_channels=128,
    activation="silu"
)

# 3. Combine them
model = TorchMD_Net(
    representation_model=representation_model,
    output_model=output_model
)

In [ ]:
import torch.nn.functional as F
from torch.optim import Adam
import gc
from tqdm import tqdm

# --- SAFETY CHECK: Force Model to GPU ---
# This fixes the "cpu and cuda:0" error
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Ensuring model is on: {device}")

# --- Setup Training ---
optimizer = Adam(model.parameters(), lr=1e-4) # removed weight_decay
loss_fn = F.l1_loss

# --- CALCULATION ---
# For Dipoles, we ONLY calculate Std Dev. We do NOT subtract Mean.
print("Calculating training set std for normalization...")
y_train = torch.cat([data.y for data in train_dataset])
y_std = y_train.std().to(device)
# y_mean is removed because 0 dipole is physically meaningful.
print(f"Training data std: {y_std.item():.4f}")

# --- Training Function ---
def train(epoch):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")

    for batch in progress_bar:
        # Move batch to GPU
        batch = batch.to(device)
        optimizer.zero_grad()

        # --- PHYSICS FIX ---
        # Only divide by std. Do NOT subtract mean.
        target = batch.y / y_std

        # Forward pass
        output = model(batch.z, batch.pos, batch.batch)

        # Handle tuple output safely
        if isinstance(output, tuple):
            pred = output[0]
        else:
            pred = output

        # Squeeze helps ensure shapes match
        loss = loss_fn(pred.squeeze(), target.squeeze())

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs
        progress_bar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(train_dataset)
    print(f"Epoch {epoch} | Avg Train Loss (Normalized): {avg_loss:.4f}")

# --- Validation Function ---
@torch.no_grad()
def validate(loader, dataset_len):
    model.eval()
    total_error = 0

    for batch in loader:
        batch = batch.to(device)
        output = model(batch.z, batch.pos, batch.batch)
        pred = output[0] if isinstance(output, tuple) else output

        pred_unscaled = pred * y_std
        error = F.l1_loss(pred_unscaled.squeeze(), batch.y.squeeze(), reduction='sum')
        total_error += error.item()

    avg_mae = total_error / dataset_len
    return avg_mae

# Usage during training: val_mae = validate(val_loader, len(val_dataset))
# Usage after training: test_mae = validate(test_loader, len(test_dataset))

# --- Start Training ---
print("Starting training...")
NUM_EPOCHS = 10

try:
    for epoch in range(1, NUM_EPOCHS + 1):
        train(epoch)
        val_mae = validate(val_loader, len(val_dataset))

    print("Training complete.")
    torch.save(model.state_dict(), "dipole_model.pt")

except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("\n[FAIL] Ran out of GPU memory!")
        torch.cuda.empty_cache()
        gc.collect()
    else:
        raise e